# What this file does
- Category vs Company analysis

# Dependencies
#### Run the following file(s) before running this code.
- 03_baseline_similarity_graph.ipynb (or 03b or 03c)
- 07_gds_Louvain_Summary.ipynb
- 21_gds_Centrality_on-graph.ipynb
- 23_gds_Company.ipynb

##### Note:
URL of the Neo4j browser:
- https://[IP address]:7473/browser/

ID & Pass: 
- Use the one in .env


In [1]:
# Config
SAMPLING:bool       = True
NUM_SAMPLE:int      = 250   # Number of sample data to be ingested to the graph database
SUMMARY_SAMPLE:int  = 20    # Number of samples as inputs of summarizing
RAND_SEED:int       = 77    # Seed for sampling
NUM_SIM:int         = 3     # Number of results from KNN search (does not include the own node)
EMBEDDING_MODEL:str = "text-embedding-3-small"
MAX_TOKENS:int      = 7800  # Max 8192 - some safety buffer about 5%
INDEX_NAME:str      = "idx:complaints_vss"
FILE_PATH:str       = "../data/original/complaints-2025-11-02_04_18.csv"

In [ ]:
import time
from datetime import datetime, timedelta

In [3]:
import os
import sys
import json
import numpy as np
import pandas as pd
from IPython.display import display
import tiktoken
import textwrap
import logging

logger = logging.getLogger("neo4j")
logger.setLevel(logging.CRITICAL)

In [ ]:
from dotenv import load_dotenv  
load_dotenv()

True

In [5]:
import neo4j

In [ ]:
# Ignore unclosed SSL socket warnings - optional in case you get these errors
import warnings

In [ ]:
warnings.filterwarnings(action="ignore", message="unclosed", category=ResourceWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning) 

In [ ]:
# Show all columns
pd.set_option('display.max_columns', None)

# Show all rows
pd.set_option('display.max_rows', None)

In [ ]:
# Timestamp (Start)
current_datetime = datetime.now()
formatted_time = current_datetime.strftime("%Y-%m-%d_%H:%M:%S")
print(formatted_time)

start_time = time.time()

2026-01-08_22:09:36


### Neo4j

In [ ]:
driver = neo4j.GraphDatabase.driver(
    uri=os.environ.get("NEO4J_URI"), 
    auth=(os.environ.get("NEO4J_USERNAME"), 
          os.environ.get("NEO4J_PASSWORD"))
)

In [ ]:
session = driver.session(database="neo4j")

In [ ]:
def my_neo4j_run_query_pandas(query, **kwargs):
    "run a query and return the results in a pandas dataframe"
    
    result = session.run(query, **kwargs)
    
    df = pd.DataFrame([r.values() for r in result], columns=result.keys())
    
    return df

# Remove Complaint nodes

In [ ]:
# query = """

# MATCH (n:Complaint)
# DETACH DELETE n;

# """

# session.run(query)

# Category-Company Analysis 

In [ ]:
# clear the old graph because degree centrality didn't work in the current graph projection
drop_query = "CALL gds.graph.drop('ds_graph', false)"

# project a fresh graph
query = """
CALL gds.graph.project(
    'ds_graph', 
    ['Category', 'Company'], 
    {
        COMPLAINS_TO: {
            type: 'COMPLAINS_TO', 
            orientation: 'NATURAL',
            properties: 'num'
        }
    }
)
"""

with driver.session() as session:
    session.run(drop_query)
    session.run(query)

In [15]:
query = """
CALL gds.degree.stream('ds_graph', {})
YIELD nodeId, score
WITH gds.util.asNode(nodeId) AS n, score
WHERE n:Category
RETURN n.summary AS category_summary, score AS number_of_companies
ORDER BY score DESC
LIMIT 10
"""

with driver.session() as session:
    result = session.run(query)
    df = pd.DataFrame([record.data() for record in result])

df.head(10)

,category_summary,number_of_companies
0,Failure to address fraudulent accounts and ina...,5.0
1,Unauthorized accounts and inaccuracies on cred...,4.0
2,Unauthorized accounts and inaccuracies on cred...,3.0
3,Fraudulent accounts on credit reports due to d...,3.0
4,Failure to correct inaccurate credit report en...,3.0
5,Identity theft resulting in fraudulent account...,3.0
6,Demand for correction of credit report inaccur...,3.0
7,Violation of consumer rights regarding credit ...,3.0
8,Identity theft resulting in fraudulent account...,3.0
9,Identity theft and fraudulent accounts on cred...,2.0


In [16]:
query = """
CALL gds.degree.stream('ds_graph', { orientation: 'REVERSE' })
YIELD nodeId, score
WITH gds.util.asNode(nodeId) AS n, score
WHERE n:Company
RETURN n.name AS company_name, score AS number_of_categories
ORDER BY score DESC
LIMIT 20
"""

with driver.session() as session:
    result = session.run(query)
    df = pd.DataFrame([record.data() for record in result])

df.head(10)

,company_name,number_of_categories
0,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",43.0
1,"EQUIFAX, INC.",42.0
2,Experian Information Solutions Inc.,41.0
3,Chime Financial Inc,3.0
4,CAPITAL ONE FINANCIAL CORPORATION,3.0
5,Resurgent Capital Services L.P.,3.0
6,"CITIBANK, N.A.",3.0
7,WELLS FARGO & COMPANY,2.0
8,"I.C. System, Inc.",2.0
9,"BANK OF AMERICA, NATIONAL ASSOCIATION",2.0


In [ ]:
# Timestamp (End)
current_datetime = datetime.now()
formatted_time = current_datetime.strftime("%Y-%m-%d_%H:%M:%S")
print(formatted_time)

end_time = time.time()
elapsed_seconds = end_time - start_time

# Convert elapsed seconds to minutes and seconds
minutes = int(elapsed_seconds // 60)
seconds = elapsed_seconds % 60

print(f"Program elapsed time: {minutes} minutes and {seconds:.2f} seconds")

2026-01-08_22:09:36
Program elapsed time: 0 minutes and 0.12 seconds
